# 12 - Word Embeddings with TensorFlow/Keras

Goal: Learn word embeddings using TensorFlow and Keras

Run with: conda activate tfenv

In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0


In [11]:
vocab = ["el", "la", "gato", "perro", "come", "duerme", "en", "casa", "corre", "nada"]
vocab_size = len(vocab)
embed_dim = 3

word_to_idx = {word: idx for idx, word in enumerate(vocab)}
print(f"Vocab size: {vocab_size}")
print(f"Words: {vocab}")
print(f"Word to index: {word_to_idx}")

Vocab size: 10
Words: ['el', 'la', 'gato', 'perro', 'come', 'duerme', 'en', 'casa', 'corre', 'nada']
Word to index: {'el': 0, 'la': 1, 'gato': 2, 'perro': 3, 'come': 4, 'duerme': 5, 'en': 6, 'casa': 7, 'corre': 8, 'nada': 9}


In [12]:
print("=== Method 1: tf.keras.layers.Embedding ===")
embedding_layer = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)

gato_idx = word_to_idx["gato"]
gato_embedding = embedding_layer(tf.constant(gato_idx))
print(f"Embedding for 'gato': {gato_embedding.numpy()}")

=== Method 1: tf.keras.layers.Embedding ===
Embedding for 'gato': [-0.02861801 -0.0093049  -0.03328844]


In [13]:
print("=== Method 2: Dense layer (manual) ===")
dense_layer = layers.Dense(embed_dim, use_bias=False)

print("\nBoth methods learn a weight matrix of shape:", embedding_layer.weights[0].shape)

=== Method 2: Dense layer (manual) ===

Both methods learn a weight matrix of shape: (10, 3)


In [14]:
import plotly.express as px
import pandas as pd

all_embeddings = embedding_layer(np.arange(vocab_size)).numpy()

df = pd.DataFrame(all_embeddings, columns=['x', 'y', 'z'])
df['word'] = vocab

fig = px.scatter_3d(df, x='x', y='y', z='z', text='word',
                    title='Word Embeddings (Random - TensorFlow)')
fig.update_traces(marker=dict(size=8), textposition='top center')
fig.show()

In [15]:
print("=== Training embeddings ===")

pairs = [
    (0, 2),
    (0, 3),
    (2, 3),
    (2, 8),
    (3, 8),
    (3, 9),
    (4, 5),
    (6, 7),
]

optimizer = keras.optimizers.Adam(learning_rate=0.1)
loss_fn = keras.losses.MeanSquaredError()

for epoch in range(500):
    total_loss = 0
    for word_idx, context_idx in pairs:
        word = tf.constant([word_idx])
        context = tf.constant([context_idx])

        with tf.GradientTape() as tape:
            word_emb = embedding_layer(word)
            context_emb = embedding_layer(context)
            loss = loss_fn(word_emb, context_emb * 0.5)

        gradients = tape.gradient(loss, embedding_layer.trainable_weights)
        optimizer.apply_gradients(zip(gradients, embedding_layer.trainable_weights))
        total_loss += loss

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss.numpy():.4f}")

=== Training embeddings ===
Epoch 0, Loss: 0.0699
Epoch 100, Loss: 0.0514
Epoch 200, Loss: 0.1856
Epoch 300, Loss: 0.0123
Epoch 400, Loss: 0.0001


In [16]:
trained_embeddings = embedding_layer(np.arange(vocab_size)).numpy()

df_trained = pd.DataFrame(trained_embeddings, columns=['x', 'y', 'z'])
df_trained['word'] = vocab

fig = px.scatter_3d(df_trained, x='x', y='y', z='z', text='word',
                    title='Word Embeddings (Trained - TensorFlow)')
fig.update_traces(marker=dict(size=8, color='red'), textposition='top center')
fig.show()

# Extra: Learn relationships from a real text

In [17]:
# Sample text in Spanish
text = """
El gato duerme en la casa. El perro corre en el jardin. 
La gata come pescado. El perro come carne.
El nino juega con la pelota. La nina lee el libro.
El pajaro vuela en el cielo. El pez nada en el agua.
La madre cocina la cena. El padre trabaja en la oficina.
El perro ladra. El gato maulla. El pajaro canta.
"""

# Tokenize and build vocabulary
words = text.lower().split()
words = [w.strip('.,') for w in words]
unique_words = list(set(words))
print(f"Text: {len(words)} words")
print(f"Unique words: {len(unique_words)}")
print(unique_words)

Text: 63 words
Unique words: 35
['madre', 'cielo', 'juega', 'oficina', 'maulla', 'agua', 'trabaja', 'ladra', 'canta', 'gata', 'casa', 'pelota', 'nina', 'con', 'vuela', 'corre', 'come', 'pescado', 'cena', 'libro', 'lee', 'la', 'padre', 'pez', 'jardin', 'pajaro', 'gato', 'nino', 'en', 'duerme', 'perro', 'el', 'nada', 'cocina', 'carne']


In [18]:
# Create word to index
text_vocab = {word: idx for idx, word in enumerate(unique_words)}
text_vocab_size = len(text_vocab)
text_embed_dim = 2

print(f"Vocabulary: {text_vocab}")

Vocabulary: {'madre': 0, 'cielo': 1, 'juega': 2, 'oficina': 3, 'maulla': 4, 'agua': 5, 'trabaja': 6, 'ladra': 7, 'canta': 8, 'gata': 9, 'casa': 10, 'pelota': 11, 'nina': 12, 'con': 13, 'vuela': 14, 'corre': 15, 'come': 16, 'pescado': 17, 'cena': 18, 'libro': 19, 'lee': 20, 'la': 21, 'padre': 22, 'pez': 23, 'jardin': 24, 'pajaro': 25, 'gato': 26, 'nino': 27, 'en': 28, 'duerme': 29, 'perro': 30, 'el': 31, 'nada': 32, 'cocina': 33, 'carne': 34}


In [19]:
# Create training pairs from context (window size = 1)
def create_pairs(words, window=1):
    pairs = []
    for i, word in enumerate(words):
        for j in range(max(0, i - window), min(len(words), i + window + 1)):
            if i != j:
                pairs.append((word, words[j]))
    return pairs

pairs = create_pairs(words)
print(f"Training pairs: {len(pairs)}")
print("Sample pairs:", pairs[:10])

Training pairs: 124
Sample pairs: [('el', 'gato'), ('gato', 'el'), ('gato', 'duerme'), ('duerme', 'gato'), ('duerme', 'en'), ('en', 'duerme'), ('en', 'la'), ('la', 'en'), ('la', 'casa'), ('casa', 'la')]


In [20]:
# Build embedding model with BATCH processing
text_embedding = layers.Embedding(input_dim=text_vocab_size, output_dim=text_embed_dim)
optimizer = keras.optimizers.Adam(learning_rate=0.05)
loss_fn = keras.losses.MeanSquaredError()

# Convert pairs to tensors and create batched dataset
word_indices = np.array([text_vocab[w] for w, c in pairs], dtype=np.int32)
context_indices = np.array([text_vocab[c] for w, c in pairs], dtype=np.int32)

# Create tf.data Dataset with batching
dataset = tf.data.Dataset.from_tensor_slices((word_indices, context_indices))
dataset = dataset.shuffle(buffer_size=len(pairs)).batch(32)

# Training with @tf.function for graph compilation (much faster)
@tf.function
def train_step(word_batch, context_batch):
    with tf.GradientTape() as tape:
        word_emb = text_embedding(word_batch)
        context_emb = text_embedding(context_batch)
        loss = loss_fn(word_emb, context_emb * 0.8)
    gradients = tape.gradient(loss, text_embedding.trainable_weights)
    optimizer.apply_gradients(zip(gradients, text_embedding.trainable_weights))
    return loss

print("Training with batch processing...")
import time

for epoch in range(500):
    start_time = time.time()
    total_loss = 0
    
    for word_batch, context_batch in dataset:
        loss = train_step(word_batch, context_batch)
        total_loss += loss
    
    epoch_time = time.time() - start_time
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss.numpy():.4f}, Time: {epoch_time:.3f}s")

Training with batch processing...
Epoch 0, Loss: 0.0136, Time: 0.694s
Epoch 100, Loss: 0.0031, Time: 0.016s
Epoch 200, Loss: 0.0034, Time: 0.011s
Epoch 300, Loss: 0.0050, Time: 0.011s
Epoch 400, Loss: 0.0026, Time: 0.011s


In [21]:
# Show learned embeddings
print("\n=== Learned Embeddings ===")
embeddings = text_embedding(np.arange(text_vocab_size)).numpy()

for word, idx in text_vocab.items():
    print(f"{word}: {embeddings[idx]}")


=== Learned Embeddings ===
madre: [-0.00982683  0.0062074 ]
cielo: [ 0.00763755 -0.00235925]
juega: [ 0.00556872 -0.00547855]
oficina: [-0.01265696  0.01149806]
maulla: [0.01917502 0.00184605]
agua: [-0.00813228  0.01651769]
trabaja: [0.00249439 0.00980364]
ladra: [0.00916414 0.01794064]
canta: [-0.01193448  0.00939545]
gata: [0.01282585 0.00526548]
casa: [-0.00981007 -0.01027759]
pelota: [0.00145048 0.00592209]
nina: [-0.00270872  0.02303376]
con: [-0.0008494   0.00104191]
vuela: [0.00141611 0.01144642]
corre: [-0.00865821 -0.00750675]
come: [-0.00531761 -0.00977397]
pescado: [0.02429522 0.00323589]
cena: [ 0.00616269 -0.0175944 ]
libro: [-0.0168307 -0.0053607]
lee: [ 0.00029245 -0.01492052]
la: [ 0.00235571 -0.00242748]
padre: [ 0.01618515 -0.00072847]
pez: [ 0.00166188 -0.02033274]
jardin: [0.00708213 0.00325705]
pajaro: [0.00806227 0.00437765]
gato: [-0.03519865 -0.0047804 ]
nino: [0.00301029 0.00215031]
en: [ 0.00330637 -0.00048616]
duerme: [0.01921433 0.00361043]
perro: [ 0.0115

In [22]:
# Visualize in 3D using Plotly
# Expand embeddings to 3D using a simple projection
embeddings_3d = np.column_stack([
    embeddings[:, 0],
    embeddings[:, 1],
    embeddings[:, 0] ** 2 + embeddings[:, 1] ** 2  # radial distance as 3rd dimension
])

df_3d = pd.DataFrame(embeddings_3d, columns=['x', 'y', 'z'])
df_3d['word'] = list(text_vocab.keys())

fig = px.scatter_3d(df_3d, x='x', y='y', z='z', text='word',
                    title='Word Embeddings from Text (3D)')
fig.update_traces(marker=dict(size=8), textposition='top center')
fig.show()

In [23]:
# Find similar words (closest embeddings)
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

def find_similar(word, top_n=3):
    word_idx = text_vocab[word]
    word_emb = embeddings[word_idx]
    
    similarities = []
    for w, idx in text_vocab.items():
        if w != word:
            sim = cosine_similarity(word_emb, embeddings[idx])
            similarities.append((w, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

print("=== Similar Words ===")
for word in ['gato', 'perro', 'come', 'casa']:
    if word in text_vocab:
        similar = find_similar(word)
        print(f"'{word}' is similar to: {similar}")

=== Similar Words ===
'gato' is similar to: [('libro', np.float32(0.9850105)), ('corre', np.float32(0.8368457)), ('casa', np.float32(0.78153))]
'perro' is similar to: [('cielo', np.float32(0.9791467)), ('juega', np.float32(0.9629518)), ('la', np.float32(0.9564464))]
'come' is similar to: [('cocina', np.float32(0.99970573)), ('carne', np.float32(0.98231566)), ('casa', np.float32(0.96539104))]
'casa' is similar to: [('carne', np.float32(0.9971502)), ('corre', np.float32(0.99554944)), ('cocina', np.float32(0.9714331))]
